### Building a company brochure generator with UI using Gradio


In [44]:
import os
print(os.getcwd())

d:\Python_Projects\llm_course\Week2


In [45]:
import sys 
sys.path.append(r"d:\Python_Projects\llm_course")

In [46]:
# import 

import os
import json
from Week1.scraper import fetch_website_links, fetch_website_contents
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [47]:
load_dotenv(override= True)

openai_api_key = os.getenv("OPENAI_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")

if openai_api_key:
    print(f"OpenAI API Key exists and begins with {openai_api_key[:8]}.....")
else:
    print("OpenAI API Key is not set.")


if google_api_key:
    print(f"Google API Key exists and begins with {google_api_key[:8]}....")
else:
    print("Google API Key not found")

openai_client = OpenAI()
gpt_model= "gpt-4.1-mini"

OpenAI API Key exists and begins with sk-proj-.....
Google API Key exists and begins with AIzaSyB-....


In [48]:
gemini_base_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

gemini = OpenAI(base_url= gemini_base_url, api_key= google_api_key)

In [49]:
links = fetch_website_links("https://edwarddonner.com")
links[:3]

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/']

### Step 1: Get relevant links.

#### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.

It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [50]:
# Generate a system prompt to fetch relevant links

link_system_prompt= """
You are provided with a list of links found on webpage.
You are able to decide which of the links would be most relevant to include
in a brochure about company, such as about page, or a company page, or 
careers/jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url.careers"}
    ]
}
"""

In [51]:
def get_link_user_prompt(url):

    user_prompt = f"""
Here are the lists of links on the website {url}.
Please decide which of these are relevant web links for a brochure about the 
company, respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (Some might be relative links)
"""
    
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)

    return user_prompt

In [52]:
print(get_link_user_prompt("http://example.com"))


Here are the lists of links on the website http://example.com.
Please decide which of these are relevant web links for a brochure about the 
company, respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (Some might be relative links)
https://iana.org/domains/example


In [53]:
def select_relevant_links(url):
    print(f"Selecting relevant links from the {url} by calling model \'{gpt_model}\'")

    response = openai_client.chat.completions.create(
        model= gpt_model,
        messages= [
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_link_user_prompt(url)}
        ],
        response_format= {'type': 'json_object'}
    )

    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links.")

    return links

In [54]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links from the https://edwarddonner.com by calling model 'gpt-4.1-mini'
Found 1 relevant links.


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'}]}

### Second step: Make the Brochure!!!
Assemble all the details into another prompt to GPT


In [55]:
def fetch_page_and_all_relevant_links(url):

    contents= fetch_website_contents(url)
    relevant_links= select_relevant_links(url)

    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"

    for link in relevant_links['links']:
        result += f"\n\n### Link: Type: {link['type']}\n"
        result += fetch_website_contents(link['url'])

    return result

In [56]:
fetch_page_and_all_relevant_links("https://huggingface.co")

Selecting relevant links from the https://huggingface.co by calling model 'gpt-4.1-mini'
Found 6 relevant links.


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


"## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nNEW\nGoogle Gemma 4 is here 💫\nStorage Buckets: AI-native object storage\nGGML and llama.cpp join Hugging Face 🔥\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\ngoogle/gemma-4-31B-it\nUpdated\n6 days ago\n•\n1.11M\n•\n1.43k\ndealignai/Gemma-4-31B-JANG_4M-CRACK\nUpdated\n4 days ago\n•\n44.2k\n•\n765\nzai-org/GLM-5.1\nUpdated\nabout 4 hours ago\n•\n1.3k\n•\n697\nnetflix/void-model\nUpdated\n2 days ago\n•\n625\ngoogle/gemma-4-26B-A4B-it\nUpdated\n6 days ago\n•\n836k\n•\n530\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nFeatured\n278\nOmniVoice\n🌍\n278\nHigh-quality voice cloning TTS for 600+ languages\nRunning\non\nZero\nMCP\n1.92k\nWan2.2 14B Preview\

In [57]:
# Creating brochure system prompt

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from 
a company website and creates a short brochure about the company for prospective 
customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the 
information.
"""

In [58]:
def get_brochure_user_prompt(company_name, url):

    user_prompt = f"""
You are looking at a company called: {company_name}.
Here are the contents of its landing pages and other relevant pages;
use this information to build a short brochure of the company in markdown 
without code blocks.\n\n
"""
    
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5000]

    return user_prompt

In [59]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links from the https://huggingface.co by calling model 'gpt-4.1-mini'
Found 5 relevant links.


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


"\nYou are looking at a company called: HuggingFace.\nHere are the contents of its landing pages and other relevant pages;\nuse this information to build a short brochure of the company in markdown \nwithout code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nNEW\nGoogle Gemma 4 is here 💫\nStorage Buckets: AI-native object storage\nGGML and llama.cpp join Hugging Face 🔥\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\ngoogle/gemma-4-31B-it\nUpdated\n6 days ago\n•\n1.11M\n•\n1.43k\ndealignai/Gemma-4-31B-JANG_4M-CRACK\nUpdated\n4 days ago\n•\n44.2k\n•\n765\nzai-org/GLM-5.1\nUpdated\nabout 4 hours ago\n•\n1.3k\n•\n697\nnetflix/void-model\nUpdated\n2 days ago\n•\n625\ngoogle/gemma-4-26B-A4

In [63]:
MODEL_MAP = {
    "GPT": "gpt-4.1-mini",
    "Gemini": "gemini-2.5-flash-lite"
}

def create_brochure(company_name, url, model):

    model_id = MODEL_MAP[model]

    if model == "GPT":
        client_con = openai_client
    elif  model == "Gemini":
        client_con = gemini
    else:
        raise ValueError("Unknown Model")

    response = client_con.chat.completions.create(
        model= model_id,
        messages= [
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ], 
        stream= True
    )

    result= ""
    for chunk in response:
        delta = chunk.choices[0].delta.content
        if delta is not None:
            result += delta
            yield result
    # for chunk in response:
    #     result += chunk.choices[0].delta.content
    #     yield result

In [64]:
import gradio as gr

In [65]:
message_input_comp_name = gr.Textbox(label= "Company name", info= "Enter your company name", lines=1)
message_input_url = gr.Textbox(label= "URL", info= "Enter the URL", lines= 2)

model_selector = gr.Dropdown(["GPT", "Gemini"], label= "Select model" ,value= "GPT")
message_output = gr.Markdown(label= "Response: ")

view= gr.Interface(
    fn= create_brochure,
    title= "BROCHURE",
    inputs= [message_input_comp_name, message_input_url, model_selector],
    outputs= [message_output],
    examples= [
        ["HuggingFace", "https://huggingface.co", "GPT"],
        ["Example", "http://example.com", "Gemini"]
    ],
    flagging_mode= "never"
)

view.launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


Selecting relevant links from the http://example.com by calling model 'gpt-4.1-mini'
Found 0 relevant links.
Selecting relevant links from the https://huggingface.co by calling model 'gpt-4.1-mini'
Found 6 relevant links.


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
